In [71]:
import  pandas as pd
from sklearn.model_selection import train_test_split
from statsmodels.tools import add_constant
from statsmodels.stats.outliers_influence import  variance_inflation_factor
import statsmodels.api as sm
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [72]:
df = pd.read_csv("/Users/mohanrajsubramaniam/PycharmProjects/python_learning/Classification Techniques for Predictive Modeling/backpain.csv")
df.head()

,pelvic_incidence,pelvic tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope,Status
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,0.744503,12.5661,14.5386,15.30468,-28.658501,43.5123,Abnormal
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,0.415186,12.8874,17.5323,16.78486,-25.530607,16.1102,Abnormal
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,0.474889,26.8343,17.4861,16.65897,-29.031888,19.2221,Abnormal
3,69.297008,24.652878,44.311238,44.644130,101.868495,11.211523,0.369345,23.5603,12.7074,11.42447,-30.470246,18.8329,Abnormal
4,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,0.543360,35.4940,15.9546,8.87237,-16.378376,24.9171,Abnormal


In [73]:
# Compute correlations with pelvic_incidence
corr = df.corr(numeric_only=True)['pelvic_incidence']
# Select variables with correlation >= 0.7 (absolute)
high_corr = corr[abs(corr) >= 0.7]
high_corr

pelvic_incidence         1.000000
lumbar_lordosis_angle    0.717282
sacral_slope             0.814960
Name: pelvic_incidence, dtype: float64

In [74]:
# Encode Status: Abnormal → 1, Normal → 0
df['Status'] = df['Status'].map({'Abnormal': 1, 'Normal': 0})
df.head()

,pelvic_incidence,pelvic tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope,Status
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,0.744503,12.5661,14.5386,15.30468,-28.658501,43.5123,1
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,0.415186,12.8874,17.5323,16.78486,-25.530607,16.1102,1
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,0.474889,26.8343,17.4861,16.65897,-29.031888,19.2221,1
3,69.297008,24.652878,44.311238,44.644130,101.868495,11.211523,0.369345,23.5603,12.7074,11.42447,-30.470246,18.8329,1
4,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,0.543360,35.4940,15.9546,8.87237,-16.378376,24.9171,1


In [75]:
# Separate features and target
X=df.drop('Status', axis=1)
X.head()

,pelvic_incidence,pelvic tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,0.744503,12.5661,14.5386,15.30468,-28.658501,43.5123
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,0.415186,12.8874,17.5323,16.78486,-25.530607,16.1102
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,0.474889,26.8343,17.4861,16.65897,-29.031888,19.2221
3,69.297008,24.652878,44.311238,44.644130,101.868495,11.211523,0.369345,23.5603,12.7074,11.42447,-30.470246,18.8329
4,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,0.543360,35.4940,15.9546,8.87237,-16.378376,24.9171


In [76]:
y=df["Status"]
y.head()

0    1
1    1
2    1
3    1
4    1
Name: Status, dtype: int64

In [77]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
y.head()

0    1
1    1
2    1
3    1
4    1
Name: Status, dtype: int64

In [78]:
# Percentage of each class in test set
percentages = y_test.value_counts(normalize=True) * 100
print(percentages)

Status
1    74.193548
0    25.806452
Name: proportion, dtype: float64


Which metric is the most appropriate metric to evaluate the model according to the problem statement?
The problem is a binary classification task where the goal is to distinguish Abnormal (1) vs Normal (0) spinal conditions.
Because the dataset is highly imbalanced (almost all rows are Abnormal), accuracy would be misleading.

The most appropriate evaluation metric is F1‑score, specifically the macro F1‑score.

F1‑score matters here because:

It balances precision and recall, which is essential when one class dominates.

It prevents a model from appearing “good” simply by predicting the majority class.

It reflects how well the model identifies the minority class (Normal), which is clinically important.

In [79]:
X_const = add_constant(X)
vif_df = pd.DataFrame({
    "feature": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])],
})
vif_df = vif_df[vif_df["feature"] != "const"]
high_vif = vif_df[vif_df["VIF"] > 5]
high_vif

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
1,pelvic_incidence,inf
2,pelvic tilt,inf
4,sacral_slope,inf


In [80]:
X = df.drop(['Status','sacral_slope'], axis=1)
X.head()

,pelvic_incidence,pelvic tilt,lumbar_lordosis_angle,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope
0,63.027817,22.552586,39.609117,98.672917,-0.254400,0.744503,12.5661,14.5386,15.30468,-28.658501,43.5123
1,39.056951,10.060991,25.015378,114.405425,4.564259,0.415186,12.8874,17.5323,16.78486,-25.530607,16.1102
2,68.832021,22.218482,50.092194,105.985135,-3.530317,0.474889,26.8343,17.4861,16.65897,-29.031888,19.2221
3,69.297008,24.652878,44.311238,101.868495,11.211523,0.369345,23.5603,12.7074,11.42447,-30.470246,18.8329
4,49.712859,9.652075,28.317406,108.168725,7.918501,0.543360,35.4940,15.9546,8.87237,-16.378376,24.9171


In [81]:
X = sm.add_constant(X)
model = sm.Logit(y,X).fit()

while True:
    pvals = model.pvalues.drop('const')
    worst = pvals.idxmax()
    if pvals.max() < 0.05:
        break
    X = X.drop(columns=[worst])
    model = sm.Logit(y, X).fit()

final_vars = list(X.columns)
final_vars.remove('const')
len(final_vars)


Optimization terminated successfully.
         Current function value: 0.284109
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.284265
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.284395
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.284820
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.285529
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.286646
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.287518
         Iterations 9
Optimization terminated successfully.
         Current function value: 0.288610
         Iterations 9


4

In [82]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=1 )
len(X_train)
len(X_test)
results = []
for depth in range(1, 9):
    model = DecisionTreeClassifier(max_depth=depth, random_state=1)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append((depth, train_acc, test_acc))
    results_df = pd.DataFrame(results, columns=['Depth','Train_Accuracy','Test_Accuracy'])
    print(results_df)

   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
1      2        0.838710       0.827957
   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
1      2        0.838710       0.827957
2      3        0.889401       0.752688
   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
1      2        0.838710       0.827957
2      3        0.889401       0.752688
3      4        0.912442       0.752688
   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
1      2        0.838710       0.827957
2      3        0.889401       0.752688
3      4        0.912442       0.752688
4      5        0.935484       0.741935
   Depth  Train_Accuracy  Test_Accuracy
0      1        0.778802       0.795699
1      2        0.838710       0.827957
2      3        0.889401       0.752688
3      4        0.912442       0.752688


In [83]:
model = DecisionTreeClassifier(max_depth=5, random_state=1)
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

degree_spondylolisthesis    0.618048
pelvic_radius               0.200816
pelvic_incidence            0.105730
pelvic tilt                 0.075406
const                       0.000000
dtype: float64

In [85]:
from sklearn.model_selection import GridSearchCV

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=1 )
param_grid = { 'max_depth': [5, 10, 15, None], 'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'] }
dt = DecisionTreeClassifier(random_state=1)
grid = GridSearchCV( estimator=dt, param_grid=param_grid, scoring='recall', cv=3 )
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)
print("Best Recall Score:", grid.best_score_)

Best Parameters: {'criterion': 'gini', 'max_depth': 15, 'splitter': 'random'}
Best Recall Score: 0.9027777777777778


Attempt #1
Feb 10, 6:15 PM
Marks: 14
Question 1
Correct Answer
Marks: 2/2

Load the dataset and identify the variables that have a correlation greater than or equal to 0.7 with the ‘pelvic_incidence’ variable.
pelvic tilt, pelvic_radius

lumbar_lordosis_angle, sacral_slope

You Selected
Direct_tilt, sacrum_angle

thoracic_slope, thoracic_slope

plt.figure(figsize=(10,5))
sns.heatmap(data.corr()[data.corr()>=0.7],annot=True,vmax=1,vmin=-1,cmap='Spectral');

Question 2
Correct Answer
Marks: 2/2

Encode Status variable: Abnormal class to 1 and Normal to 0.

Split the data into a 70:30 ratio. What is the percentage of 0 and 1 classes in the test data (y_test)?

Note - Do not use stratify on the dataset.

1: In a range of 0.1 to 0.2

0: In a range of 0.2 to 0.3

1: In a range of 0.5 to 0.6

0: In a range of 0.3 to 0.6

1: In a range of  0.6 to 0.7

0: In a range of  0.3 to 0.4

1: In a range of 0.7 to 0.8

0: In a range of  0.2 to 0.3

You Selected
data['Status'] = data['Status'].apply(lambda x: 1 if x=='Abnormal' else 0)
X = data.drop(['Status'], axis=1)
Y = data['Status']
#Splitting data in train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.30, random_state = 1)
y_test.value_counts(normalize=True)

Question 3
Incorrect Answer
Marks: 0/2

Which metric is the most appropriate metric to evaluate the model according to the problem statement?
Accuracy

Recall

Correct Option
Precision

F1 score

You Selected
Predicting a person doesn't have an abnormal spine and a person has an abnormal spine - A person who needs treatment will be missed. Hence, reducing such false negatives is important

Question 4
Correct Answer
Marks: 2/2

Check for multicollinearity in data and choose the variables which show high multicollinearity. (VIF value greater than 5)
sacrum_angle, pelvic tilt, sacral_slope

pelvic_slope, cervical_tilt, sacrum_angle

pelvic_incidence, pelvic tilt, sacral_slope

You Selected
pelvic_incidence, pelvic tilt, lumbar_lordosis_angle

# dataframe with numerical column only
num_feature_set = X.copy()
num_feature_set = add_constant(num_feature_set)
num_feature_set = num_feature_set.astype(float)
# Calculating VIF
vif_series = pd.Series([variance_inflation_factor(num_feature_set.values,i) for i in range(num_feature_set.shape[1])],index=num_feature_set.columns, dtype = float)
print('Series before feature selection: \n\n{}\n'.format(vif_series))

Question 5
Correct Answer
Marks: 2/2

How many minimum numbers of attributes will we need to drop to remove multicollinearity (or get a VIF value less than 5) from the data?
1

You Selected
2

3

4

# Dropping first variable with high VIF
num_feature_set1 = num_feature_set.drop(['pelvic_incidence'],axis=1)
# Checking VIF value
vif_series1 = pd.Series([variance_inflation_factor(num_feature_set1.values,i) for i in range(num_feature_set1.shape[1])],index=num_feature_set1.columns, dtype = float)
print('Series before feature selection: \n\n{}\n'.format(vif_series1))
# Dropping second variable with high VIF
num_feature_set2 = num_feature_set.drop(['pelvic tilt'],axis=1)
# Checking VIF value
vif_series2 = pd.Series([variance_inflation_factor(num_feature_set2.values,i) for i in range(num_feature_set2.shape[1])],index=num_feature_set2.columns, dtype = float)
print('Series before feature selection: \n\n{}\n'.format(vif_series2))
# Dropping the third variable with high VIF
num_feature_set3 = num_feature_set.drop(['sacral_slope'],axis=1)
# Checking VIF value
vif_series3 = pd.Series([variance_inflation_factor(num_feature_set3.values,i) for i in range(num_feature_set3.shape[1])],index=num_feature_set3.columns, dtype = float)
print('Series before feature selection: \n\n{}\n'.format(vif_series3))

# Dropping any of the 3 variables will result in a VIF value of less than 5, therefore the minimum number of attributes to be dropped is 1.

Question 6
Correct Answer
Marks: 2/2

Drop sacral_slope attribute and proceed to build a logistic regression model. Drop all the insignificant variables and keep only significant variables (p-value < 0.05).

How many significant variables are left in the final model excluding the constant?

1

2

3

4

You Selected
# Dropping sacral slope
X_train, X_test, y_train, y_test = train_test_split(num_feature_set3, Y, test_size=0.30, random_state = 1)
# Iteratively dropping variables with a high p-value
X_train2 = X_train.drop(['pelvic_slope'],axis=1)
X_test2 = X_test.drop(['pelvic_slope'],axis=1)
logit = sm.Logit(y_train, X_train2.astype(float))
lg = logit.fit()
print(lg.summary())
X_train3 = X_train2.drop(['scoliosis_slope'],axis=1)
X_test3 = X_test2.drop(['scoliosis_slope'],axis=1)
logit = sm.Logit(y_train, X_train3.astype(float))
lg = logit.fit()
print(lg.summary())
X_train4 = X_train3.drop(['cervical_tilt'],axis=1)
X_test4 = X_test3.drop(['cervical_tilt'],axis=1)
logit = sm.Logit(y_train, X_train4.astype(float))
lg = logit.fit()
print(lg.summary())
X_train5 = X_train4.drop(['Direct_tilt'],axis=1)
X_test5 = X_test4.drop(['Direct_tilt'],axis=1)
logit = sm.Logit(y_train, X_train5.astype(float))
lg = logit.fit()
print(lg.summary())
X_train6 = X_train5.drop(['lumbar_lordosis_angle'],axis=1)
X_test6 = X_test5.drop(['lumbar_lordosis_angle'],axis=1)
logit = sm.Logit(y_train, X_train6.astype(float))
lg = logit.fit()
print(lg.summary())
X_train7 = X_train6.drop(['sacrum_angle'],axis=1)
X_test7 = X_test6.drop(['sacrum_angle'],axis=1)
logit = sm.Logit(y_train, X_train7.astype(float))
lg = logit.fit()
print(lg.summary())
X_train8 = X_train7.drop(['thoracic_slope'],axis=1)
X_test8 = X_test7.drop(['thoracic_slope'],axis=1)
logit = sm.Logit(y_train, X_train8.astype(float))
lg = logit.fit()
print(lg.summary())

Question 7
Incorrect Answer
Marks: 0/2

Select the correct option for the following:

Train a decision tree model with default parameters and vary the depth from 1 to 8 (both values included) and compare the model performance at each value of depth

At depth = 1, the decision tree gives the highest recall among all the models on the training set.

At depth = 2, the decision tree gives the highest recall among all the models on the training set.

At depth = 5, the decision tree gives the highest recall among all the models on the training set.

You Selected
At depth = 8, the decision tree gives the highest recall among all the models on the training set.

Correct Option
score_DT = []
for i in range(1,9):
 dTree = DecisionTreeClassifier(max_depth=i,criterion = 'gini', random_state=1)
 dTree.fit(X_train, y_train)
 pred = dTree.predict(X_train)
 case = {'Depth':i,'Recall':recall_score(y_train,pred)}
 score_DT.append(case)
print(score_DT)

Question 8
Correct Answer
Marks: 2/2

Plot the feature importance of the variables given by the model which gives the maximum value of recall on the training set in Q7. Which are the 2 most important variables respectively?
lumbar_lordosis_angle, sacrum_angle

degree_spondylolisthesis, pelvic tilt

You Selected
scoliosis_slope, cervial_tilt

Direct_tilt, pelvic_radius

Question 9
Incorrect Answer
Marks: 0/2

Perform hyperparmater tuning for Decision tree using GridSrearchCV.

Use the following list of hyperparameters and their values:

Maximum depth: [5,10,15, None],
criterion: ['gini','entropy'],
splitter: ['best','random']
Set cv = 3 in grid search
Set scoring = 'recall' in grid search
Which of the following statements is/are True?

A) GridSeachCV selects the max_depth as 10
B) GridSeachCV selects the criterion as 'gini'
C) GridSeachCV selects the splitter as 'random'
D) GridSeachCV selects the splitter as 'best'
E) GridSeachCV selects the max_depth as 5
F) GridSeachCV selects the criterion as 'entropy'

A, B, and C

B, C, and E

You Selected
A, C, and F

Correct Option
D, E, and F

# Choose the type of classifier.
estimator = DecisionTreeClassifier(random_state=1)
# Grid of parameters to choose from
parameters = {'max_depth': [5,10,15,None],
 'criterion' : ['gini','entropy'],
 'splitter' : ['best','random']
 }

# Run the grid search
grid_obj = GridSearchCV(estimator, parameters, scoring='recall',cv=3)
grid_obj = grid_obj.fit(X_train, y_train)
# Set the clf to the best combination of parameters
estimator = grid_obj.best_estimator_
# Fit the best algorithm to the data.
estimator.fit(X_train, y_train)

Question 10
Correct Answer
Marks: 2/2

Compare the model performance of a Decision Tree with default parameters and the tuned Decision tree built in Q9 on the test set.

Which of the following statements is/are True?

A) Recall Score of tuned model > Recall Score of decision tree with default parameters
B) Recall Score of tuned model < Recall Score of decision tree with default parameters
C) F1 Score of tuned model > F1 Score Score of decision tree with default parameters
D) F1 Score of tuned model < F1 Score of decision tree with default parameters

A and B

B and C

C and D

A and D

You Selected
# Training decision tree with default parameters
model = DecisionTreeClassifier(random_state=1)
model.fit(X_train,y_train)
# Tuned model
estimator.fit(X_train, y_train)
# Predicting on the test set
y_pred_test1 = model.predict(X_test)
y_pred_test2 = estimator.predict(X_test)
# Checking model performance of Decision Tree with default parameters
print(recall_score(y_test,y_pred_test1))
print(metrics.f1_score(y_test,y_pred_test1))
# Checking model performance of tunedDecision Tree
print(recall_score(y_test,y_pred_test2))
print(metrics.f1_score(y_test,y_pred_test2))
